# SimJEB 1 - data, QA and splits

Downloads SimJEB from Harvard Dataverse, profiles it, builds the cached graph dataset,
and freezes the train/validation/test splits.

**Runs on CPU.** Nothing here needs a GPU, and the accelerator quota is worth saving
for training.

**Settings required:** Internet **ON** (Settings -> Internet), Accelerator **None**.
Run with *Save & Run All (Commit)* so a browser disconnect cannot kill it.

Output: `/kaggle/working/graphs/*.pt` (~2.4 GB) plus the QA report and split files.
Save the version, then add its output as an input dataset to notebook 2.

## Setup

Point `REPO` at the project. Options in order of convenience:

1. push the repo to GitHub and clone it (below),
2. or upload the `src/` folder as a Kaggle Dataset and set `REPO` to its path.

In [ ]:
GITHUB_REPO = "https://github.com/Vedavamsi-3/simjeb-structural-gnn.git"  # e.g. "https://github.com/<you>/simjeb-structural-gnn.git"
REPO = "/kaggle/working/simjeb-structural-gnn"

import subprocess, sys, os
from pathlib import Path

if GITHUB_REPO and not Path(REPO).exists():
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO, REPO], check=True)

# PyTorch Geometric is not pre-installed on Kaggle. Since PyG 2.x it is pure Python,
# so this is a plain install -- no compiling torch-scatter against the CUDA build.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch-geometric", "meshio", "trimesh"], check=True)

sys.path.insert(0, REPO)
os.chdir(REPO)
print("repo:", REPO)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch

from src.data.fetch import download_all, SimJEBSource
from src.data.make_dataset import make_dataset, dataset_summary
from src.data.split import (
    load_metadata, leakage_groups, official_split, select_grouped_split, verify_split,
)
from src.qa import alignment, outliers, plots

ARCHIVES = Path("/kaggle/temp/archives")   # not saved as output; large
SCRATCH  = Path("/kaggle/temp/scratch")
WORK     = Path("/kaggle/working")
GRAPHS   = WORK / "graphs"
QA       = WORK / "qa"
SPLITS   = WORK / "splits"
for d in (ARCHIVES, SCRATCH, GRAPHS, QA / "figures", SPLITS):
    d.mkdir(parents=True, exist_ok=True)

## 1. Download

~7 GB of archives. They are **never fully extracted** -- expanded they exceed 20 GB,
and Kaggle's working directory holds about that. Files are read one member at a time.

Dataverse resets long connections, so the download resumes from a partial file rather
than restarting. Safe to re-run.

In [ ]:
%%time
paths = download_all(ARCHIVES)
for key, path in paths.items():
    print(f"{key:12s} {path.stat().st_size/1e9:6.2f} GB  {path.name}")

## 2. Profile the dataset before training on it

SimJEB is crowd-sourced CAD -- 381 designs by different people, meshed and solved
automatically. A single corrupt model distorts the normalisation statistics that every
other model is scaled by, so this runs first.

Most of the statistics are already published in `all_bracket_metadata.tab`, so this is
a 155 KB read rather than a 5 GB scan.

In [ ]:
meta = load_metadata(ARCHIVES / "all_bracket_metadata.tab")
print(f"{len(meta)} models")
print(meta["category"].value_counts().to_string())
print()
print("peak vertical stress: min %.0f  mean %.0f  max %.0f MPa"
      % (meta.max_ver_stress.min(), meta.max_ver_stress.mean(), meta.max_ver_stress.max()))
print("genus:                min %.0f  max %.0f" % (meta.genus.min(), meta.genus.max()))

In [ ]:
report = outliers.detect(meta)
print(outliers.summarise(report, meta))

plots.plot_metric_distributions(meta, report, QA / "figures" / "metric_distributions.png")
plots.plot_stress_context(meta, QA / "figures" / "stress_vs_yield.png")

from IPython.display import Image, display
display(Image(str(QA / "figures" / "stress_vs_yield.png")))

### Frame check

SimJEB states the models are pre-aligned. This measures it, using the four bolt holes
and the load lug as landmarks -- the parts that are standardised across designs.

Translation and rotation are treated differently on purpose. A translation changes
nothing and is normalised away during graph building. A rotation is not neutral: the
load is a fixed global vector that does **not** rotate with the part, so a turned
bracket under the same global load is a different problem, not a different view.
Those models are excluded, never re-oriented -- rotating the geometry back would not
rotate the load that produced the stress field, leaving an aligned mesh whose answers
came from a different loading.

In [ ]:
%%time
from src.data.parse_fem import parse_fem, landmark_centroids

source = SimJEBSource.open(ARCHIVES)
model_ids = source.model_ids()
print(f"{len(model_ids)} models with all three files present")

landmarks = {}
failed_landmarks = {}
for n, mid in enumerate(model_ids, 1):
    try:
        files = source.extract_model(mid, SCRATCH / str(mid))
        deck = parse_fem(files["fem"])
        coords = pd.read_csv(files["csv"], usecols=["x", "y", "z"]).to_numpy()
        landmarks[mid] = landmark_centroids(deck, coords, spc_set=1, load_set=2)
    except Exception as exc:
        failed_landmarks[mid] = f"{type(exc).__name__}: {exc}"
    finally:
        import shutil; shutil.rmtree(SCRATCH / str(mid), ignore_errors=True)
    if n % 50 == 0:
        print(f"  {n}/{len(model_ids)}", flush=True)

print(f"landmarks for {len(landmarks)} models; {len(failed_landmarks)} failed")

In [ ]:
fits, rotation_outliers = alignment.assess_alignment(landmarks)

rot = np.array([f.rotation_deg for f in fits])
tra = np.array([f.translation_mm for f in fits])
rms = np.array([f.rmsd_mm for f in fits])
print(f"rotation    max {rot.max():8.3f} deg   median {np.median(rot):.4f}")
print(f"translation max {tra.max():8.3f} mm    median {np.median(tra):.4f}")
print(f"residual    max {rms.max():8.3f} mm    median {np.median(rms):.4f}")
print(f"reflected models: {sum(f.is_reflected for f in fits)}")
print(f"\nrotation outliers to exclude: {rotation_outliers}")

plots.plot_alignment(fits, QA / "figures" / "alignment.png")
plots.plot_landmarks(landmarks, QA / "figures" / "landmarks.png")
display(Image(str(QA / "figures" / "alignment.png")))
display(Image(str(QA / "figures" / "landmarks.png")))

### Exclusions

Only two things are excluded: **hard failures** (physically impossible -- negative
genus meaning a disconnected mesh, non-finite results, a load that produced no
movement) and **rotation outliers**.

Statistical flags are *kept*. Dropping every unusual model removes exactly the hard
cases and quietly inflates the test score; they are recorded so the tail of the
results can be cross-referenced against them later.

In [ ]:
excluded = {}
for mid, reason in report.hard_failures.items():
    excluded[int(mid)] = reason
for mid in rotation_outliers:
    excluded[int(mid)] = "rotation outlier: load frame does not rotate with the part"
for mid, reason in failed_landmarks.items():
    excluded[int(mid)] = f"could not parse: {reason}"

pd.DataFrame(
    [{"model_id": k, "reason": v} for k, v in sorted(excluded.items())]
).to_csv(QA / "excluded.csv", index=False)

pd.DataFrame({
    "model_id": [f.model_id for f in fits],
    "rotation_deg": rot, "translation_mm": tra, "rmsd_mm": rms,
    "reflected": [f.is_reflected for f in fits],
}).to_csv(QA / "alignment_report.csv", index=False)

pd.DataFrame({
    "model_id": report.statistical,
    "flagged_by_z": [m in report.by_method["modified_z"] for m in report.statistical],
    "flagged_by_forest": [m in report.by_method["isolation_forest"] for m in report.statistical],
}).to_csv(QA / "flagged.csv", index=False)

print(f"excluded {len(excluded)} models:")
for mid, reason in sorted(excluded.items()):
    print(f"  {mid}: {reason}")
print(f"\nflagged for review but KEPT: {len(report.statistical)}")

## 3. Build the cached graphs

One `.pt` per model, ~6 MB each. Geometry is stored once with the targets for all four
load cases alongside -- the three unused cases cost ~1.2 MB per model and mean
switching load case later never needs this stage re-run.

Restartable: existing outputs are skipped, so a failure part-way does not discard the
work already done.

In [ ]:
%%time
keep = [m for m in model_ids if m not in excluded]
print(f"building {len(keep)} of {len(model_ids)} models\n")

build = make_dataset(ARCHIVES, GRAPHS, scratch_dir=SCRATCH, model_ids=keep)
build.save(QA / "build_report.json")

In [ ]:
summary = dataset_summary(GRAPHS)
print(f"models        : {summary['n_models']}")
print(f"surface nodes : {summary['surface_nodes']['min']:,} to "
      f"{summary['surface_nodes']['max']:,}  (mean {summary['surface_nodes']['mean']:,.0f})")
print(f"total nodes   : {summary['surface_nodes']['total']:,}")
print(f"disk          : {summary['disk_gb']} GB")

## 4. Splits

Two, and both are reported.

**Official** -- SimJEB ships `test_split_0/1/2`. Using one keeps results comparable
with published work. They are random over models though: 9 of the 24 same-submission
design groups straddle train and test in split 0, and category shares range from 8.7%
to 30.2% against a 20% target.

**Grouped** -- built here: no design family straddles the boundary, categories
balanced. The stricter number.

The seed is chosen from ten candidates by smallest worst-case standardised mean
difference. That is legitimate *only* because the criterion is fixed before any model
is trained -- it is recorded in the split file for audit.

In [ ]:
built_ids = set(build.built) | set(build.skipped)
usable = meta[meta["id"].isin(built_ids)]
groups = leakage_groups(usable)

sizes = pd.Series(list(groups.values())).value_counts().value_counts().sort_index()
print("leakage group sizes:")
for size, count in sizes.items():
    print(f"  {count:3d} groups of {size}")

official = official_split(usable, groups, index=0)
grouped  = select_grouped_split(usable, groups, seeds=range(10))

for split in (official, grouped):
    v = verify_split(split, usable, groups)
    split.verification = v
    split.save(SPLITS / f"{split.name}.json")
    print(f"\n{split.name}")
    print(f"  train/val/test    : {v['n_train']}/{v['n_val']}/{v['n_test']}")
    print(f"  groups straddling : {v['groups_straddling']}")
    print(f"  max SMD           : {v['smd_max']:.3f}  ({'balanced' if v['smd_max']<0.1 else 'above the 0.1 threshold'})")

In [ ]:
for split in (official, grouped):
    plots.plot_category_balance(usable, split, QA / "figures" / f"balance_{split.name}.png")
    display(Image(str(QA / "figures" / f"balance_{split.name}.png")))

## 5. Hand off

Everything notebook 2 needs is now in `/kaggle/working`. Save this version, then in
notebook 2 add **this notebook's output** as an input dataset.

In [ ]:
import shutil
shutil.rmtree(ARCHIVES, ignore_errors=True)   # ~7 GB, not worth saving as output
shutil.rmtree(SCRATCH, ignore_errors=True)

print("ready for notebook 2:")
print(f"  graphs  {len(list(GRAPHS.glob('*.pt')))} files")
for p in sorted(SPLITS.glob('*.json')):
    print(f"  split   {p.name}")
for p in sorted(QA.glob('*.csv')):
    print(f"  qa      {p.name}")